In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [24]:
import warnings
import json
import joblib
from datetime import datetime, timezone

import xgboost as xgb
import lightgbm as lgb

from sklearn.base import clone
from sklearn.metrics import classification_report, cohen_kappa_score
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from imblearn.ensemble import BalancedRandomForestClassifier
from scripts.helpers import save_classification_report
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [25]:
df           = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv")
df_nlp       = pd.read_csv(PROJECT_ROOT / "working_data" / "nlp_oof_logits_probs.csv")
df_emergency = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_emergency_keyword_flags_matched_only.csv")
print("Data loaded:", df.shape, df_nlp.shape, df_emergency.shape)


Data loaded: (58124, 49) (58124, 8) (58124, 15)


In [26]:
print("Applying Cyclical Encoding to Time Features...")

if 'arrival_time' in df.columns:
    df['arrival_time'] = np.where(df['arrival_time'] < 0, np.nan, df['arrival_time'])
    hours   = np.floor_divide(df['arrival_time'], 100)
    minutes = np.mod(df['arrival_time'], 100)
    df['arrival_hour_float'] = hours + (minutes / 60.0)
    df['arrival_hour']       = hours
    df['is_shift_change']    = (
        ((df['arrival_hour_float'] >= 6)  & (df['arrival_hour_float'] < 8)) |
        ((df['arrival_hour_float'] >= 18) & (df['arrival_hour_float'] < 20))
    ).astype(int)

cycle_maxes = {'visit_month': 12.0, 'day_of_week': 7.0, 'arrival_hour_float': 24.0}
for col, max_val in cycle_maxes.items():
    if col in df.columns:
        angle = 2 * np.pi * df[col] / max_val
        df[f'{col}_sin'] = np.sin(angle)
        df[f'{col}_cos'] = np.cos(angle)

if 'day_of_week' in df.columns:
    max_dow      = df['day_of_week'].max()
    weekend_days = [5, 6] if max_dow <= 6 else [6, 7]
    is_weekend   = df['day_of_week'].isin(weekend_days)
    if 'arrival_hour_float' in df.columns:
        df['is_weekend_night'] = (is_weekend & (df['arrival_hour_float'] >= 18)).astype(int)

cols_to_drop = ['visit_month', 'day_of_week', 'arrival_time', 'arrival_hour_float']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Dataframe shape after cyclical encoding: {df.shape}")


Applying Cyclical Encoding to Time Features...
Dataframe shape after cyclical encoding: (58124, 55)


In [27]:
print("Adding clinical ratios and vital missingness...")

df["shock_index"]         = df["heart_rate"] / df["sys_bp"].replace(0, np.nan)
df["shock_index_age_adj"] = df["shock_index"] * np.where(df["age"] >= 65, 1.2, 1.0)
df["map"]                 = (df["sys_bp"] + 2 * df["dias_bp"]) / 3
df["pulse_pressure"]      = df["sys_bp"] - df["dias_bp"]
df["age_hr_interaction"]  = df["age"] * df["heart_rate"]
df["resp_spo2_ratio"]     = df["resp_rate"] / df["spo2"].replace(0, np.nan)
df["elderly_tachy"]       = ((df["age"] >= 65) & (df["heart_rate"] > 100)).astype(int)

history_cols = [c for c in df.columns if "hist_" in c]
if history_cols:
    df["history_count"] = df[history_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

news2 = pd.Series(0, index=df.index, dtype=float)
if "resp_rate" in df.columns:
    rr = df["resp_rate"]
    news2 += np.select([rr <= 8, (rr>=9)&(rr<=11), (rr>=12)&(rr<=20), (rr>=21)&(rr<=24), rr>=25], [3, 1, 0, 2, 3], default=0)
if "spo2" in df.columns:
    sp = df["spo2"]
    news2 += np.select([sp<=91, (sp>=92)&(sp<=93), (sp>=94)&(sp<=95), sp>=96], [3, 2, 1, 0], default=0)
if "sys_bp" in df.columns:
    sbp = df["sys_bp"]
    news2 += np.select([sbp<=90, (sbp>=91)&(sbp<=100), (sbp>=101)&(sbp<=110), (sbp>=111)&(sbp<=219), sbp>=220], [3, 2, 1, 0, 3], default=0)
df["news2_score"] = news2

vital_cols = [c for c in ["heart_rate","sys_bp","dias_bp","resp_rate","spo2","temp"] if c in df.columns]
if vital_cols:
    df["vital_missing"] = df[vital_cols].isna().any(axis=1).astype(int)
    for col in vital_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(df[num_cols].median())


Adding clinical ratios and vital missingness...


In [28]:
if "row_index" in df_nlp.columns:
    df_nlp = df_nlp.sort_values("row_index").set_index("row_index").reindex(df.index).reset_index(drop=True)

df_nlp       = df_nlp.drop(columns=["row_index", "fold", "year"], errors="ignore")
df_emergency = df_emergency.drop(columns=["row_index"], errors="ignore")

if len(df_nlp) != len(df):
    raise ValueError("NLP features are not aligned.")
print("NLP alignment OK:", df_nlp.shape)


NLP alignment OK: (58124, 5)


In [29]:
cols_to_drop = ["intervention_iv_fluids", "vitals_during_visit", "wait_time_minutes", "year", "target_triage_acuity", "chief_complaint_text", "injury_cause_text", "true_class", "pred_class"]
X_tabular   = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()
year_groups = df["year"].copy()

def map_esi_to_group(val):
    if val in (1, 2): return 0
    if val == 3:      return 1
    if val in (4, 5): return 2
    return np.nan

y_ord       = df["target_triage_acuity"].map(map_esi_to_group).dropna().astype(int)
X_tabular   = X_tabular.loc[y_ord.index].copy()
year_groups = year_groups.loc[y_ord.index].copy()

def to_category(df_in):
    df_out = df_in.copy()
    for col in df_out.select_dtypes(include=["object", "string"]).columns:
        df_out[col] = df_out[col].astype("category")
    return df_out

X_tabular = to_category(X_tabular)
hist_cols = [c for c in X_tabular.columns if c.startswith("hist")]
if hist_cols: X_tabular[hist_cols] = X_tabular[hist_cols].fillna(0).astype(int).astype(bool)

X_nlp       = df_nlp.loc[y_ord.index].copy()
X_emergency = to_category(df_emergency.loc[y_ord.index].copy())

em_rename = {c: f"em_{c}" for c in X_emergency.columns if c in X_tabular.columns}
if em_rename: X_emergency = X_emergency.rename(columns=em_rename)
X_tabular = pd.concat([X_tabular.reset_index(drop=True), X_emergency.reset_index(drop=True)], axis=1)

bias_columns = {"residence", "region", "race", "no_payment", "insurance"}
X_tabular    = X_tabular.drop(columns=bias_columns, errors="ignore")

def category_codes(df_in):
    df_out = df_in.copy()
    for col in df_out.select_dtypes(include=["category"]).columns:
        df_out[col] = df_out[col].cat.codes
    return df_out.fillna(-1)

X_tabular_codes = category_codes(X_tabular)
X_nlp_num       = X_nlp[[c for c in X_nlp.columns if "prob" in c]].apply(pd.to_numeric, errors="coerce").fillna(0)
n_classes       = int(y_ord.nunique())
print(f"Features: {X_tabular.shape}, NLP: {X_nlp_num.shape}")


Features: (58124, 73), NLP: (58124, 3)


In [30]:
years_sorted    = np.sort(year_groups.unique())
year_bins       = np.array_split(years_sorted, 3)
year_bucket_map = {yr: idx for idx, years in enumerate(year_bins) for yr in years}
year_bucket     = year_groups.map(year_bucket_map)
splitter        = GroupKFold(n_splits=3)
print("Splitting on year buckets:", {i: list(b) for i, b in enumerate(year_bins)})


Splitting on year buckets: {0: [np.int64(2018), np.int64(2019)], 1: [np.int64(2020), np.int64(2021)], 2: [np.int64(2022)]}


In [31]:
def qwk_score(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def compute_sample_weights(y_train):
    classes    = np.unique(y_train)
    weights    = compute_class_weight("balanced", classes=classes, y=y_train)
    weight_map = dict(zip(classes, weights))
    return np.array([weight_map[v] for v in y_train])

def fit_cutpoints(y_true, y_pred, n_cls):
    df_cut  = pd.DataFrame({"y": y_true, "pred": y_pred})
    medians = df_cut.groupby("y")["pred"].median().sort_index()
    if len(medians) < n_cls: return np.quantile(y_pred, np.linspace(0, 1, n_cls + 1)[1:-1])
    return np.array([(medians.iloc[i] + medians.iloc[i + 1]) / 2.0 for i in range(n_cls - 1)])

def apply_cutpoints(y_pred, cutpoints):
    return np.digitize(y_pred, cutpoints, right=False)

def print_evaluation_report(y_true, y_pred, name):
    print(f"\n{'='*50}\n  {name} – OOF Evaluation\n  QWK: {qwk_score(y_true, y_pred):.4f}\n{'='*50}")
    print(classification_report(y_true, y_pred, digits=4, target_names=["Urgent(0)", "Emergent(1)", "NonUrgent(2)"]))

def regressor_soft_proba(train_pred, val_pred, y_train, n_cls):
    centers = pd.Series(train_pred).groupby(y_train.values).median()
    if len(centers) < n_cls: centers = pd.Series(np.linspace(train_pred.min(), train_pred.max(), n_cls), index=range(n_cls))
    centers = centers.reindex(range(n_cls)).interpolate().to_numpy()
    tau     = np.var(train_pred) + 1e-6
    dist    = (val_pred[:, None] - centers[None, :]) ** 2
    probs   = np.exp(-dist / (2.0 * tau))
    return probs / probs.sum(axis=1, keepdims=True)


In [32]:
xgb_reg = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist", enable_categorical=True)
lgb_reg = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63, subsample=1.0, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1)
brf     = BalancedRandomForestClassifier(n_estimators=500, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1)

xgb_cls = xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist", enable_categorical=True)
lgb_cls = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, num_leaves=63, subsample=1.0, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1)


In [33]:
base_oof = {
    "xgb_reg": np.zeros((len(y_ord), n_classes)),
    "lgb_reg": np.zeros((len(y_ord), n_classes)),
    "brf":     np.zeros((len(y_ord), n_classes)),
    "xgb_cls": np.zeros((len(y_ord), n_classes)),
    "lgb_cls": np.zeros((len(y_ord), n_classes)),
}

models_dir = PROJECT_ROOT / "results" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(splitter.split(X_tabular, y_ord, groups=year_bucket), start=1):
    print(f"\nFold {fold}...")
    X_tr,  X_val  = X_tabular.iloc[train_idx],       X_tabular.iloc[val_idx]
    Xc_tr, Xc_val = X_tabular_codes.iloc[train_idx],  X_tabular_codes.iloc[val_idx]
    y_tr,  y_val  = y_ord.iloc[train_idx],           y_ord.iloc[val_idx]
    sw             = compute_sample_weights(y_tr)

    # 1. Regressors (with cutpoints for QWK tracking)
    for name, model in [("xgb_reg", xgb_reg), ("lgb_reg", lgb_reg)]:
        m = clone(model).fit(X_tr, y_tr, sample_weight=sw)
        tr_preds, val_preds = m.predict(X_tr), m.predict(X_val)
        base_oof[name][val_idx] = regressor_soft_proba(tr_preds, val_preds, y_tr, n_classes)
        cp       = fit_cutpoints(y_tr, tr_preds, n_classes)
        fold_qwk = qwk_score(y_val, apply_cutpoints(val_preds, cp))
        print(f"  {name:15} fold {fold} QWK: {fold_qwk:.4f}")
        joblib.dump(m, models_dir / f"{name}_fold{fold}.joblib")

    # 2. BRF
    m = clone(brf).fit(Xc_tr, y_tr, sample_weight=sw)
    base_oof["brf"][val_idx] = m.predict_proba(Xc_val)
    tr_preds_p  = m.predict_proba(Xc_tr) @ np.arange(n_classes)
    val_preds_p = m.predict_proba(Xc_val) @ np.arange(n_classes)
    cp          = fit_cutpoints(y_tr, tr_preds_p, n_classes)
    print(f"  brf             fold {fold} QWK: {qwk_score(y_val, apply_cutpoints(val_preds_p, cp)):.4f}")
    joblib.dump(m, models_dir / f"brf_fold{fold}.joblib")

    # 3. Classifiers (no cutpoints, just argmax for QWK tracking)
    for name, model in [("xgb_cls", xgb_cls), ("lgb_cls", lgb_cls)]:
        m = clone(model).fit(X_tr, y_tr, sample_weight=sw)
        probs = m.predict_proba(X_val)
        base_oof[name][val_idx] = probs
        fold_qwk = qwk_score(y_val, np.argmax(probs, axis=1))
        print(f"  {name:15} fold {fold} QWK: {fold_qwk:.4f}")
        joblib.dump(m, models_dir / f"{name}_fold{fold}.joblib")

# Add NLP as a "base model" for averaging
base_oof["nlp"] = X_nlp_num.to_numpy()
print("\nBase OOF ready:", sorted(base_oof.keys()))



Fold 1...
  xgb_reg         fold 1 QWK: 0.4040
  lgb_reg         fold 1 QWK: 0.3915
  brf             fold 1 QWK: 0.3769
  xgb_cls         fold 1 QWK: 0.3992
  lgb_cls         fold 1 QWK: 0.3785

Fold 2...
  xgb_reg         fold 2 QWK: 0.4148
  lgb_reg         fold 2 QWK: 0.4094
  brf             fold 2 QWK: 0.3876
  xgb_cls         fold 2 QWK: 0.4062
  lgb_cls         fold 2 QWK: 0.3982

Fold 3...
  xgb_reg         fold 3 QWK: 0.4099
  lgb_reg         fold 3 QWK: 0.4117
  brf             fold 3 QWK: 0.3934
  xgb_cls         fold 3 QWK: 0.4084
  lgb_cls         fold 3 QWK: 0.4021

Base OOF ready: ['brf', 'lgb_cls', 'lgb_reg', 'nlp', 'xgb_cls', 'xgb_reg']


In [34]:
print("\nFinal OOF Reports:")
class_values = np.arange(n_classes)
for name, probs in base_oof.items():
    preds = probs @ class_values
    cp    = fit_cutpoints(y_ord, preds, n_classes)
    print_evaluation_report(y_ord, apply_cutpoints(preds, cp), name)



Final OOF Reports:

  xgb_reg – OOF Evaluation
  QWK: 0.3998
              precision    recall  f1-score   support

   Urgent(0)     0.3309    0.6424    0.4368      9443
 Emergent(1)     0.5905    0.2778    0.3778     29568
NonUrgent(2)     0.5029    0.6810    0.5785     19113

    accuracy                         0.4696     58124
   macro avg     0.4747    0.5337    0.4644     58124
weighted avg     0.5195    0.4696    0.4534     58124


  lgb_reg – OOF Evaluation
  QWK: 0.3917
              precision    recall  f1-score   support

   Urgent(0)     0.3297    0.6392    0.4350      9443
 Emergent(1)     0.5846    0.2758    0.3748     29568
NonUrgent(2)     0.4967    0.6723    0.5713     19113

    accuracy                         0.4652     58124
   macro avg     0.4703    0.5291    0.4604     58124
weighted avg     0.5143    0.4652    0.4492     58124


  brf – OOF Evaluation
  QWK: 0.3949
              precision    recall  f1-score   support

   Urgent(0)     0.3205    0.6435    0.42

In [35]:
print("Simple Average Meta-Learner (Regs + Cls + NLP)...")
avg_probs = np.mean(np.array(list(base_oof.values())), axis=0)
avg_pred  = avg_probs @ np.arange(n_classes)
avg_cp    = fit_cutpoints(y_ord, avg_pred, n_classes)
print_evaluation_report(y_ord, apply_cutpoints(avg_pred, avg_cp), "Simple Average Meta-Learner")


Simple Average Meta-Learner (Regs + Cls + NLP)...

  Simple Average Meta-Learner – OOF Evaluation
  QWK: 0.4505
              precision    recall  f1-score   support

   Urgent(0)     0.3514    0.6563    0.4577      9443
 Emergent(1)     0.6211    0.3275    0.4288     29568
NonUrgent(2)     0.5408    0.7045    0.6119     19113

    accuracy                         0.5049     58124
   macro avg     0.5044    0.5628    0.4995     58124
weighted avg     0.5509    0.5049    0.4937     58124



In [36]:
print("Weighted Average Meta-Learner (weight = OOF QWK)...")
cv = np.arange(n_classes)
weights = []
names   = []
for name, probs in base_oof.items():
    preds = probs @ cv
    cp    = fit_cutpoints(y_ord, preds, n_classes)
    qwk   = qwk_score(y_ord, apply_cutpoints(preds, cp))
    weights.append(max(qwk, 0.01))
    names.append(name)

print("  Weights:", dict(zip(names, [round(w, 4) for w in weights])))
w_probs = np.average(np.array(list(base_oof.values())), axis=0, weights=np.array(weights))
w_pred  = w_probs @ cv
w_cp    = fit_cutpoints(y_ord, w_pred, n_classes)
print_evaluation_report(y_ord, apply_cutpoints(w_pred, w_cp), "Weighted Average Meta-Learner")


Weighted Average Meta-Learner (weight = OOF QWK)...
  Weights: {'xgb_reg': 0.3998, 'lgb_reg': 0.3917, 'brf': 0.3949, 'xgb_cls': 0.4071, 'lgb_cls': 0.3991, 'nlp': 0.474}

  Weighted Average Meta-Learner – OOF Evaluation
  QWK: 0.4562
              precision    recall  f1-score   support

   Urgent(0)     0.3539    0.6569    0.4600      9443
 Emergent(1)     0.6260    0.3363    0.4376     29568
NonUrgent(2)     0.5465    0.7065    0.6163     19113

    accuracy                         0.5102     58124
   macro avg     0.5088    0.5666    0.5046     58124
weighted avg     0.5557    0.5102    0.5000     58124



In [37]:
print("Stacked Logistic Regression (All probes + original features)...")
meta_X = np.hstack(list(base_oof.values()))
meta_oof = np.zeros(len(y_ord), dtype=int)
cv = np.arange(n_classes)

for fold, (train_idx, val_idx) in enumerate(splitter.split(meta_X, y_ord, groups=year_bucket), start=1):
    X_mtr, X_mval = meta_X[train_idx], meta_X[val_idx]
    y_tr,  y_val  = y_ord.iloc[train_idx], y_ord.iloc[val_idx]
    
    m = LogisticRegression(solver="saga", max_iter=800, C=0.25, penalty="l2", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
    m.fit(X_mtr, y_tr, sample_weight=compute_sample_weights(y_tr))
    
    tr_p, val_p = m.predict_proba(X_mtr) @ cv, m.predict_proba(X_mval) @ cv
    cp = fit_cutpoints(y_tr, tr_p, n_classes)
    meta_oof[val_idx] = apply_cutpoints(val_p, cp)
    print(f"  Meta fold {fold} QWK: {qwk_score(y_val, meta_oof[val_idx]):.4f}")
    joblib.dump(m, models_dir / f"stacked_meta_fold{fold}.joblib")

print_evaluation_report(y_ord, meta_oof, "Stacked Logistic Meta-Learner")


Stacked Logistic Regression (All probes + original features)...
  Meta fold 1 QWK: 0.5133
  Meta fold 2 QWK: 0.5070
  Meta fold 3 QWK: 0.5159

  Stacked Logistic Meta-Learner – OOF Evaluation
  QWK: 0.5135
              precision    recall  f1-score   support

   Urgent(0)     0.3934    0.6548    0.4915      9443
 Emergent(1)     0.6875    0.4750    0.5618     29568
NonUrgent(2)     0.6216    0.7147    0.6649     19113

    accuracy                         0.5830     58124
   macro avg     0.5675    0.6148    0.5728     58124
weighted avg     0.6180    0.5830    0.5843     58124



In [ ]:
# ── Train and Save Final Models on Full Dataset ──
print("\nTraining final models on full dataset...")
sw_full = compute_sample_weights(y_ord)

final_models = [
    ("xgb_reg", xgb_reg),
    ("lgb_reg", lgb_reg),
    ("brf",     brf),
    ("xgb_cls", xgb_cls),
    ("lgb_cls", lgb_cls),
]

for name, model in final_models:
    print(f"  Fitting final {name}...")
    if name == "brf":
        m_final = clone(model).fit(X_tabular_codes, y_ord, sample_weight=sw_full)
    else:
        m_final = clone(model).fit(X_tabular, y_ord, sample_weight=sw_full)
    
    save_path = models_dir / f"{name}_final.joblib"
    joblib.dump(m_final, save_path)
    print(f"  Saved to: {save_path}")




Training final models on full dataset...
  Fitting final xgb_reg...
  Saved to: /home/gaurav/python/kaggle/triage/results/models/xgb_reg_final.joblib
  Fitting final lgb_reg...
  Saved to: /home/gaurav/python/kaggle/triage/results/models/lgb_reg_final.joblib
  Fitting final brf...
  Saved to: /home/gaurav/python/kaggle/triage/results/models/brf_final.joblib
  Fitting final xgb_cls...
  Saved to: /home/gaurav/python/kaggle/triage/results/models/xgb_cls_final.joblib
  Fitting final lgb_cls...
  Saved to: /home/gaurav/python/kaggle/triage/results/models/lgb_cls_final.joblib

All final models trained and saved successfully.


In [39]:

print("\nTraining final meta-learner on full OOF dataset...")
final_meta_lr = LogisticRegression(solver="saga", max_iter=800, C=0.25, penalty="l2", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)
final_meta_lr.fit(meta_X, y_ord, sample_weight=compute_sample_weights(y_ord))
meta_save_path = models_dir / "stacked_meta_final.joblib"
joblib.dump(final_meta_lr, meta_save_path)
print(f"  Saved meta-learner to: {meta_save_path}")

print("\nAll final models trained and saved successfully.")



Training final meta-learner on full OOF dataset...
  Saved meta-learner to: /home/gaurav/python/kaggle/triage/results/models/stacked_meta_final.joblib

All final models trained and saved successfully.
